# IS353 - Lab 4: Centrality và PageRank

**MSSV:** 23520588

Đồ thị chính là Zachary Karate Club (34 đỉnh, hai phe quanh Mr. Hi và Officer). ego-Facebook để ở mục cuối, đề ghi là tùy chọn vì betweenness trên hơn 4000 đỉnh chạy lâu.

Các độ đo:

- **Degree:** số hàng xóm.
- **Betweenness:** tỉ lệ đường đi ngắn nhất đi qua đỉnh.
- **Eigenvector:** đỉnh quan trọng nếu nối với đỉnh quan trọng.
- **PageRank:** eigenvector của bước đi ngẫu nhiên có nhảy teleport.


## Cài thư viện


In [ ]:
%pip install networkx matplotlib numpy scipy


## Import


In [ ]:
%matplotlib inline

from pathlib import Path
import gzip
import shutil
import urllib.request

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## Hàm dùng chung


In [ ]:
def lab_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "BT" / "Lab4",
        Path(r"d:/Study/IS353/BT/Lab4"),
    ]
    for folder in candidates:
        if (folder / "requirement.txt").exists():
            return folder
    raise FileNotFoundError("Không thấy requirement.txt của Lab 4.")


LAB = lab_dir()


def top_nodes(score, k=5):
    return sorted(score, key=score.get, reverse=True)[:k]


def print_top(score, name, k=5):
    print(f"=== Top {k} by {name} ===")
    print(f"{'Rank':<6}{'Node':<8}{'Score':>12}")
    for rank, node in enumerate(top_nodes(score, k), start=1):
        print(f"{rank:<6}{str(node):<8}{score[node]:>12.4f}")


def centrality_bundle(G):
    degree = dict(G.degree())
    betweenness = nx.betweenness_centrality(G)
    eigenvector = nx.eigenvector_centrality(G, max_iter=1000)
    # alpha của NetworkX là xác suất đi theo cạnh, không phải xác suất teleport.
    pagerank = nx.pagerank(G, alpha=0.85)
    return {
        "degree": degree,
        "betweenness": betweenness,
        "eigenvector": eigenvector,
        "pagerank": pagerank,
    }


## Karate Club

`nx.karate_club_graph()` đánh số 0 đến 33. Đỉnh 0 là Mr. Hi (huấn luyện viên), đỉnh 33 là Officer (chủ tịch). Câu lạc bộ sau đó tách thành hai phe quanh hai người này.


In [ ]:
karate = nx.karate_club_graph()
print(f"Karate Club: {karate.number_of_nodes()} nodes, {karate.number_of_edges()} edges")
print("Mr. Hi:", [n for n, d in karate.nodes(data=True) if d["club"] == "Mr. Hi"][:8], "...")
scores = centrality_bundle(karate)


## 1. Compare rankings: What are the top-5 nodes by degree, betweenness, eigenvector, and PageRank? Why do they differ?

### Also answer for the Karate Club graph. Which measure makes the most sense for this social context?

### 1. So sánh thứ hạng: 5 đỉnh cao nhất theo degree, betweenness, eigenvector, và PageRank là gì? Vì sao chúng khác nhau? Trả lời cho cả đồ thị Karate Club. Độ đo nào hợp lý nhất cho ngữ cảnh xã hội này?


In [ ]:
for name in ("degree", "betweenness", "eigenvector", "pagerank"):
    print_top(scores[name], name)

print()
print(f"{'Node':<8}{'Club':<12}{'Degree':>8}{'Betweenness':>14}{'Eigenvector':>14}{'PageRank':>12}")
for node in range(karate.number_of_nodes()):
    in_any_top = any(node in top_nodes(scores[name]) for name in scores)
    if not in_any_top:
        continue
    print(
        f"{node:<8}{karate.nodes[node]['club']:<12}"
        f"{scores['degree'][node]:>8}"
        f"{scores['betweenness'][node]:>14.4f}"
        f"{scores['eigenvector'][node]:>14.4f}"
        f"{scores['pagerank'][node]:>12.4f}"
    )


Top 5 của Karate Club, PageRank với xác suất đi theo cạnh 0.85:

| Hạng | Degree | Betweenness | Eigenvector | PageRank |
| --- | --- | --- | --- | --- |
| 1 | 33 (17) | 0 (0.438) | 33 (0.373) | 33 (0.097) |
| 2 | 0 (16) | 33 (0.304) | 0 (0.355) | 0 (0.089) |
| 3 | 32 (12) | 32 (0.145) | 2 (0.317) | 32 (0.076) |
| 4 | 2 (10) | 2 (0.144) | 32 (0.309) | 2 (0.063) |
| 5 | 1 (9) | 31 (0.138) | 1 (0.266) | 1 (0.057) |

Bốn danh sách không trùng nhau vì mỗi độ đo đếm một kiểu "quan trọng" khác nhau.

Degree chỉ đếm số bạn. 33 và 0 đứng đầu vì họ quen nhiều người nhất trong câu lạc bộ.

Betweenness đếm số lần một đỉnh nằm trên đường đi ngắn nhất giữa các cặp khác. 0 nhích lên hạng 1 vì nhiều đường nối hai phe đi qua ông Hi. Đỉnh 31 lọt top 5 betweenness dù bậc chỉ là 6, không có trong top 5 degree.

Eigenvector cho điểm cao khi hàng xóm cũng có điểm cao. Đỉnh 2 vượt 32 so với thứ tự degree, vì 2 nằm trong vùng dày quanh ông Hi chứ không chỉ có nhiều cạnh.

PageRank cùng ý tưởng với eigenvector, nhưng người đi ngẫu nhiên thỉnh thoảng nhảy tới một đỉnh bất kỳ. Nhờ bước nhảy đó, thứ tự PageRank bám sát degree hơn: 33, 0, 32, 2, 1.

Với câu chuyện câu lạc bộ này, degree hoặc PageRank hợp nhất. Câu chuyện là hai lãnh đạo, Mr. Hi (0) và Officer (33), kéo theo hai phe. Hai độ đo đó đều đặt đúng hai người này lên đầu. Betweenness hợp hơn nếu câu hỏi là ai đứng giữa hai phe, không phải ai là thủ lĩnh.


## 2. Bridges: Which nodes have high betweenness but not high degree? What role do they play?

### 2. Cầu nối: Đỉnh nào có betweenness cao nhưng degree không cao? Chúng đóng vai trò gì?


In [ ]:
degree_rank = {node: rank for rank, node in enumerate(top_nodes(scores["degree"], karate.number_of_nodes()), start=1)}
between_rank = {node: rank for rank, node in enumerate(top_nodes(scores["betweenness"], karate.number_of_nodes()), start=1)}

print(f"{'Node':<8}{'Degree':>8}{'Deg rank':>10}{'Btw rank':>10}{'Betweenness':>14}{'Club'}")
for node in top_nodes(scores["betweenness"], 10):
    if degree_rank[node] <= 5:
        continue
    print(
        f"{node:<8}{scores['degree'][node]:>8}{degree_rank[node]:>10}{between_rank[node]:>10}"
        f"{scores['betweenness'][node]:>14.4f}  {karate.nodes[node]['club']}"
    )


Đỉnh có betweenness cao mà degree không cao: 31 (bậc 6, hạng degree 7, hạng betweenness 5), 8 (bậc 5, hạng betweenness 6), 13 (bậc 5), 19 (bậc 3, hạng degree 19 nhưng hạng betweenness 9), 5 và 6 (bậc 4).

Họ không phải hub. Họ nằm trên nhiều đường đi ngắn giữa các nhóm, nên đóng vai cầu nối: bỏ họ đi thì hai phía phải đi đường vòng dài hơn. Đỉnh 19 là ví dụ rõ nhất, rất ít bạn nhưng vẫn đứng ở hạng 9 betweenness.


## 3. Random surfer model: How does teleportation probability alpha (0.50 -> 0.95) affect PageRank on the Karate Club graph?

### 3. Mô hình random surfer: Xác suất teleport alpha (0.50 -> 0.95) ảnh hưởng đến PageRank trên đồ thị Karate Club như thế nào?

Trong đề, alpha là xác suất teleport. Trong NetworkX, tham số `alpha` là xác suất đi theo cạnh. Cell dưới truyền `alpha = 1 - teleport`.


In [ ]:
teleports = [0.50, 0.65, 0.80, 0.95]
print(f"{'Teleport':<12}{'Top 5 nodes':<28}{'Max - min':>12}")
for teleport in teleports:
    rank = nx.pagerank(karate, alpha=1 - teleport)
    top = top_nodes(rank, 5)
    spread = max(rank.values()) - min(rank.values())
    print(f"{teleport:<12}{str(top):<28}{spread:>12.4f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
for teleport in teleports:
    rank = nx.pagerank(karate, alpha=1 - teleport)
    ordered = top_nodes(rank, karate.number_of_nodes())
    ax.plot(range(1, 11), [rank[node] for node in ordered[:10]], marker="o", label=f"teleport={teleport:.2f}")
ax.set_xlabel("Rank")
ax.set_ylabel("PageRank")
ax.set_title("Karate Club PageRank khi tăng xác suất teleport")
ax.legend()
fig.tight_layout()
fig.savefig(LAB / "karate_pagerank_alpha.png", bbox_inches="tight")
plt.show()


Khi alpha (xác suất nhảy ngẫu nhiên) tăng từ 0.50 lên 0.95, điểm PageRank bị dồn về đều. Chênh lệch giữa đỉnh cao nhất và thấp nhất giảm từ khoảng 0.062 xuống 0.008, trong khi điểm đều hoàn toàn là 1/34, khoảng 0.029.

Thứ tự đầu vẫn là 33, 0, 32. Từ teleport 0.65 trở đi, đỉnh 1 vượt đỉnh 2: lúc người đi nhảy lung tung nhiều hơn, cấu trúc cục bộ quanh ông Hi bớt quyết định điểm số, số bạn trực tiếp quan trọng hơn. Teleport cao làm PageRank gần với degree và gần với phân bố đều.


## 4. PageRank: In what real-world scenario would you prefer PageRank over eigenvector centrality?

### 4. PageRank: Trong tình huống thực tế nào bạn sẽ chọn PageRank thay vì eigenvector centrality?

PageRank hợp hơn eigenvector khi đồ thị có hướng và không đi lòng vòng mãi được. Ví dụ rõ nhất là web: trang A trỏ tới trang B, nhiều trang không có link đi ra, và đồ thị không liên thông mạnh. Eigenvector trên đồ thị như vậy có thể không hội tụ, hoặc cho cả một nhóm trang điểm 0 vì không có đường đi tới.

Teleport của PageRank bảo đảm từ mọi trang đều nhảy được sang trang khác, nên mọi trang đều có điểm dương. Cùng lý do, PageRank hợp với mạng theo dõi (follow), trích dẫn, hoặc luồng tiền, nơi một liên kết có chiều và một nút có thể không trỏ đi đâu cả.


## 5. Export the Karate Club graph to Gephi, run PageRank and betweenness there, and take a screenshot of the visualization.

### 5. Xuất đồ thị Karate Club ra Gephi, chạy PageRank và betweenness ở đó, rồi chụp màn hình hình vẽ.


In [ ]:
export = karate.copy()
nx.set_node_attributes(export, {n: float(v) for n, v in scores["betweenness"].items()}, "betweenness")
nx.set_node_attributes(export, {n: float(v) for n, v in scores["pagerank"].items()}, "pagerank")
nx.set_node_attributes(export, {n: int(v) for n, v in scores["degree"].items()}, "degree")
gexf_path = LAB / "karate_club.gexf"
nx.write_gexf(export, gexf_path)
print(f"Exported: {gexf_path}")


File `karate_club.gexf` mang theo thuộc tính degree, betweenness và pagerank. Trong Gephi, layout Fruchterman-Reingold tách hai phe. Màu nút theo betweenness làm các cầu nối nổi lên. Màu theo pagerank làm 0 và 33 nổi lên. Ảnh chụp để trong thư mục Lab 4 cùng notebook.


## 6. (Optional) Run and answer the above questions on a large real-world dataset, such as the ego-Facebook graph.

### 6. (Tùy chọn) Chạy và trả lời các câu trên với một tập dữ liệu thực tế lớn, chẳng hạn đồ thị ego-Facebook.

Cùng bốn độ đo trên `facebook_combined.txt`. Betweenness duyệt đường đi từ mọi đỉnh, mất vài phút đến vài chục phút.


In [ ]:
def load_facebook() -> nx.Graph:
    names = [
        LAB / "facebook_combined.txt",
        LAB.parent / "Lab1_2" / "facebook_combined.txt",
        LAB.parent / "Lab3" / "facebook_combined.txt",
        Path.cwd() / "facebook_combined.txt",
    ]
    for path in names:
        if path.exists():
            print(f"Đọc {path}")
            return nx.read_edgelist(path, create_using=nx.Graph)
    gz_path = LAB / "facebook_combined.txt.gz"
    txt_path = LAB / "facebook_combined.txt"
    url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
    print(f"Tải {url}")
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, "rb") as src, open(txt_path, "wb") as dst:
        shutil.copyfileobj(src, dst)
    return nx.read_edgelist(txt_path, create_using=nx.Graph)


facebook = load_facebook()
print(f"Facebook: {facebook.number_of_nodes()} nodes, {facebook.number_of_edges()} edges")
fb_scores = centrality_bundle(facebook)
for name in ("degree", "betweenness", "eigenvector", "pagerank"):
    print_top(fb_scores[name], name)

fb_degree_rank = {
    node: rank
    for rank, node in enumerate(top_nodes(fb_scores["degree"], facebook.number_of_nodes()), start=1)
}
print()
print("Betweenness cao, degree không nằm top 10:")
for node in top_nodes(fb_scores["betweenness"], 10):
    if fb_degree_rank[node] > 10:
        print(
            f"node {node}: degree {fb_scores['degree'][node]} "
            f"(hạng degree {fb_degree_rank[node]}), "
            f"betweenness {fb_scores['betweenness'][node]:.4f}"
        )


Top 5 ego-Facebook (4039 đỉnh, 88234 cạnh), PageRank với xác suất đi theo cạnh 0.85:

| Hạng | Degree | Betweenness | Eigenvector | PageRank |
| --- | --- | --- | --- | --- |
| 1 | 107 (1045) | 107 (0.481) | 1912 (0.096) | 3437 (0.0076) |
| 2 | 1684 (792) | 1684 (0.338) | 2266 (0.087) | 107 (0.0069) |
| 3 | 1912 (755) | 3437 (0.236) | 2206 (0.086) | 1684 (0.0064) |
| 4 | 3437 (547) | 1912 (0.229) | 2233 (0.085) | 0 (0.0063) |
| 5 | 0 (347) | 1085 (0.149) | 2464 (0.084) | 1912 (0.0039) |

Degree và betweenness cùng chỉ các hub lớn: 107, 1684, 1912, 3437. PageRank cũng nằm trong nhóm này nhưng đảo thứ tự, 3437 lên đầu dù bậc chỉ hạng 4. Eigenvector khác hẳn: 2266, 2206, 2233, 2464 không nằm trong top degree. Độ đo này khóa vào một cụm dày quanh 1912, vì hàng xóm của họ cũng nằm trong cụm đó.

Cầu nối rõ nhất là 1085: bậc 66, hạng degree 851, nhưng hạng betweenness 5. 698 (bậc 68), 567 (bậc 63), 58 (bậc 12) cũng vậy. 860 chỉ có bậc 2 nhưng betweenness vẫn cao. Đó là đỉnh nằm giữa các cụm, không phải người có nhiều bạn.


![Karate Club trên Gephi](karate_club_gephi.png)

Karate Club trên Gephi. Màu và kích thước nút theo degree. Hai nút lớn là 0 (Mr. Hi) và 33 (Officer).